In [ ]:
import sys
import subprocess

def install_gnn_dependencies():
    # Detect Python version
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    print(f"Detected Python {version}")

    # Map Python versions to compatible Torch/PyG versions
    if version == "3.12":
        torch_ver = "2.2.0"
    else:
        torch_ver = "2.1.0"
    
    # Define the index URL for the specific binaries
    pyg_index = f"https://data.pyg.org/whl/torch-{torch_ver}+cpu.html"
    torch_index = "https://download.pytorch.org/whl/cpu"

    print(f"Executing prioritized installation for Torch {torch_ver}...")

    try:
        # Step 1: Install Torch FIRST and ALONE
        # This ensures the foundation is solid before anything else is added
        print("Step 1: Prioritizing Torch installation...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--user",
                              f"torch=={torch_ver}", "--extra-index-url", torch_index])

        # Step 2: Install NumPy (resolves the "Numpy is not available" runtime error)
        print("Step 2: Installing NumPy 1.26.4...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "--upgrade",
                              "numpy==1.26.4"])

        # Step 3: Install Scatter and Sparse Binaries with --user flag
        # These REQUIRE torch to be fully installed to see the headers
        print("Step 3: Installing GNN Extensions (Scatter/Sparse)...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--user",
                              "torch-scatter", "torch-sparse", "-f", pyg_index])

        # Step 4: Install remaining project requirements
        print("Step 4: Installing remaining requirements...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", 
                              "-r", "requirements.txt"])
        
        print("\n✓ Environment successfully configured with Torch prioritized.")
        
    except subprocess.CalledProcessError as e:
        print(f"\n× Installation failed at step {e.cmd} with error code {e.returncode}.")
        print("Ensure you have internet access and sufficient disk quota.")

install_gnn_dependencies()

Detected Python 3.10
Installing Foundations (NumPy/Torch 2.1.0) and matching GNN extensions...
Step 1: Installing NumPy 1.26.4 and Torch...
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu
Step 2: Installing GNN Extensions (Scatter/Sparse)...
Looking in links: https://data.pyg.org/whl/torch-2.1.0+cpu.html
Step 3: Installing remaining requirements...
Ignoring torch: markers 'python_version >= "3.12"' don't match your environment
Ignoring torch-scatter: markers 'python_version >= "3.12"' don't match your environment
Ignoring torch-sparse: markers 'python_version >= "3.12"' don't match your environment
  Using cached torch_sparse-0.6.17.tar.gz (209 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'

× Installation failed with error code 1.
Try running the commands manually in the terminal 

  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [17 lines of output]
      Traceback (most recent call last):
        File "/student/minalex/.local/lib/python3.10/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 389, in <module>
          main()
        File "/student/minalex/.local/lib/python3.10/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
        File "/student/minalex/.local/lib/python3.10/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_settings)
        File "/tmp/pip-build-env-pddp1ub6/overlay/local/lib/python3.10/dist-packages/setuptools/build_meta.py", line 333, in get_requires_for_build_wheel
          return self._get_build_requires(config_settings, requirements=[])
    

# Download banksim dataset from kaggle

**Citation:** Lopez-Rojas, Edgar Alonso; Axelsson, Stefan. *Banksim: A bank payments simulator for fraud detection research*. Inproceedings 26th European Modeling and Simulation Symposium, EMSS 2014, Bordeaux, France, pp. 144–152, Dime University of Genoa, 2014, ISBN: 9788897999324. https://www.researchgate.net/publication/265736405_BankSim_A_Bank_Payment_Simulation_for_Fraud_Detection_Research

In [3]:
import kagglehub
import os
import shutil
from csv_to_gzip import csv_to_gzip

# Check for required files in banksim directory
files_to_check = [
    "banksim/bs140513_032310.csv.gz",
    "banksim/bsNET140513_032310.csv.gz"
]

files_exist = all(os.path.exists(f) for f in files_to_check)

if files_exist:
    print("Both required files found. Skipping download.")
else:
    print("Required files missing. Downloading dataset...")
    path = kagglehub.dataset_download("ealaxi/banksim1")
    shutil.copytree(path, "banksim", dirs_exist_ok=True)
    print("Downloaded dataset to 'banksim'.")
    
    # Compress CSV files to gzip
    for csv_file in ['bs140513_032310.csv', 'bsNET140513_032310.csv']:
        csv_path = os.path.join("banksim", csv_file)
        if os.path.exists(csv_path):
            csv_to_gzip(csv_path)
            print(f"Compressed {csv_file} to {csv_file}.gz")
            os.remove(csv_path)

print("Path to dataset files:", "banksim")

Required files missing. Downloading dataset...
Downloaded dataset to 'banksim'.
Compressing: banksim/bs140513_032310.csv
Output: banksim/bs140513_032310.csv.gz
Original size: 48,986,035 bytes
Compressed size: 7,075,076 bytes
Compression ratio: 85.6%
Successfully created: banksim/bs140513_032310.csv.gz
Compressed bs140513_032310.csv to bs140513_032310.csv.gz
Compressing: banksim/bsNET140513_032310.csv
Output: banksim/bsNET140513_032310.csv.gz
Original size: 32,665,953 bytes
Compressed size: 6,219,961 bytes
Compression ratio: 81.0%
Successfully created: banksim/bsNET140513_032310.csv.gz
Compressed bsNET140513_032310.csv to bsNET140513_032310.csv.gz
Path to dataset files: banksim


# Download the scotiabank data files

In [4]:
import gdown
import zipfile
import os
from csv_to_gzip import csv_to_gzip

data_path = "data"
# Check for required files in data directory
files_to_check = [
    "data/labels.csv.gz",
    "data/kyc_individual.csv.gz",
    "data/kyc_smallbusiness.csv.gz",
    "data/card.csv.gz"
]

files_exist = all(os.path.exists(f) for f in files_to_check)
files_exist = files_exist and os.path.isdir(data_path) and len(os.listdir(data_path)) == 12

if files_exist:
    print("Required files found. Skipping download.")
else:
    print("Required files missing. Downloading dataset...")
    
    # Download data.zip
    url = "https://drive.google.com/file/d/1A7w3GqZTCVsv-A8gXj5NKV2FNc0zoduG/view?usp=sharing"
    zip_path = "data.zip"

    print("Downloading data.zip...")
    gdown.download(url, zip_path, fuzzy=True)
    print("Download complete.")

    # Extract the zip file
    print("Extracting data.zip...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("data")
    print("Extraction complete.")

    # Compress all CSV files in the data directory to .csv.gz
    print("Compressing CSV files to gzip...")
    for filename in os.listdir(data_path):
        if filename.endswith('.csv'):
            csv_path = os.path.join(data_path, filename)
            csv_to_gzip(csv_path)
            print(f"Compressed {filename} to {filename}.gz")
            os.remove(csv_path)

    # Clean up the zip file
    os.remove(zip_path)
    print("Cleaned up data.zip")

print("Data files ready in 'data' directory")


Required files missing. Downloading dataset...


Downloading...
From (original): https://drive.google.com/uc?id=1A7w3GqZTCVsv-A8gXj5NKV2FNc0zoduG
From (redirected): https://drive.google.com/uc?id=1A7w3GqZTCVsv-A8gXj5NKV2FNc0zoduG&confirm=t&uuid=d2829247-df6d-416e-9945-e7a76a2c5648
To: /student/minalex/imi-bigdata-2026/data.zip
100%|██████████| 114M/114M [00:01<00:00, 60.0MB/s] 


Download complete.
Extracting data.zip...
Extraction complete.
Compressing CSV files to gzip...
Compressing: data/kyc_individual.csv
Output: data/kyc_individual.csv.gz
Original size: 4,035,647 bytes
Compressed size: 975,576 bytes
Compression ratio: 75.8%
Successfully created: data/kyc_individual.csv.gz
Compressed kyc_individual.csv to kyc_individual.csv.gz
Compressing: data/kyc_occupation_codes.csv
Output: data/kyc_occupation_codes.csv.gz
Original size: 4,268 bytes
Compressed size: 1,733 bytes
Compression ratio: 59.4%
Successfully created: data/kyc_occupation_codes.csv.gz
Compressed kyc_occupation_codes.csv to kyc_occupation_codes.csv.gz
Compressing: data/cheque.csv
Output: data/cheque.csv.gz
Original size: 12,558,384 bytes
Compressed size: 3,079,795 bytes
Compression ratio: 75.5%
Successfully created: data/cheque.csv.gz
Compressed cheque.csv to cheque.csv.gz
Compressing: data/abm.csv
Output: data/abm.csv.gz
Original size: 13,889,321 bytes
Compressed size: 3,379,367 bytes
Compression r

# GraphSAGE — Inductive Fraud Detection

### Design Choices for Inductive Learning

Rather than passing transaction features into the convolution (GAT-style), we **pre-aggregate transaction features into the customer node** itself. This means any new customer node can be fully described using only their own KYC data + leakage-safe transaction aggregates — no need to re-embed the whole graph.

**Customer node (13 features):**
- KYC: `age, income, tenure, sales, emp_count, is_biz`
- Transaction aggregates: `avg_amount, max_amount, std_amount, txn_count, cash_rate, ecom_rate, avg_24h_velocity`

**Customer labels:** derived on the customer node itself (`fraud_label`), using BankSim any-fraud-per-customer logic and Scotiabank customer labels.

**Category / City nodes:** learned embeddings (initialized to one-hot, updated during training)

**Graph edges:** customer → category, customer → city (bidirectional via `ToUndirected`)


# Prepare transaction nodes for Graph Attention Network

Combine the BankSim and Scotiabank transactions into a unified master pool aligned to the card schema:

**Schema**
[transaction_id, customer_id, amount_cad, debit_credit, transaction_datetime, merchant_category, ecommerce_ind, cash_indicator, country, province, city, source_dataset, time_delta, velocity_24h, cust_idx, cat_idx, city_idx]

**Field definitions**
- **transaction_id**: Unique transaction identifier (BankSim uses a sequential row id).
- **customer_id**: Unique customer identifier.
- **amount_cad**: Transaction amount in CAD (BankSim amounts multiplied by 1.62 from EUR).
- **debit_credit**: Debit/credit flag (BankSim uses `debit`; Scotiabank uses source values).
- **transaction_datetime**: Transaction timestamp (BankSim derived from `step`).
- **merchant_category**: Mapped industry category or transfer type.
- **ecommerce_ind**: 1 if e-commerce (e.g., tech/content), else 0.
- **cash_indicator**: 1 for cash-like activity (ABM), else 0.
- **country/province/city**: Location fields (BankSim defaults to Canada/Ontario/Toronto; transfers use N/A).
- **source_dataset**: Origin dataset (banksim, scotia_card, scotia_abm, scotia_cheque/eft/emt/westernunion/wire).
- **time_delta**: Minutes since the customer’s previous transaction (0 for first transaction).
- **velocity_24h**: Count of the customer’s prior transactions in the trailing 24 hours (current transaction excluded to avoid time travel).
- **cust_idx/cat_idx/city_idx**: Integer-encoded indices for graph nodes.


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import gc

banksim_path = "banksim"
data_path = "data"

MASTER_POOL_REQUIRED_COLS = {
    'transaction_id', 'customer_id', 'amount_cad', 'debit_credit',
    'transaction_datetime', 'merchant_category', 'ecommerce_ind',
    'cash_indicator', 'country', 'province', 'city', 'source_dataset',
    'time_delta', 'velocity_24h', 'cust_idx', 'cat_idx', 'city_idx'
}


def get_mapping(series):
    unique_vals = series.unique()
    return {val: i for i, val in enumerate(unique_vals)}, len(unique_vals)


def compute_temporal_customer_features(master_pool):
    master_pool = master_pool.sort_values(
        ['customer_id', 'transaction_datetime', 'transaction_id']
    ).copy()

    master_pool['time_delta'] = (
        master_pool.groupby('customer_id')['transaction_datetime']
        .diff()
        .dt.total_seconds()
        .div(60)
        .fillna(0)
        .astype('float32')
    )
    print("Computed time delta")

    print("Computing 24h customer velocity (prior-only)...")
    master_pool = master_pool.set_index('transaction_datetime')
    master_pool['velocity_24h'] = (
        master_pool.groupby('customer_id')['amount_cad']
        .rolling(window='24h', closed='left')
        .count()
        .reset_index(level=0, drop=True)
        .fillna(0)
        .astype('float32')
    )
    master_pool = master_pool.reset_index()
    return master_pool


def build_master_transaction_pool(banksim_path, data_path):
    def shrink_and_categorize(df):
        for col in df.select_dtypes(include=['object']):
            df[col] = df[col].astype('category')
        for col in df.select_dtypes(include=['float']):
            df[col] = pd.to_numeric(df[col], downcast='float')
        for col in df.select_dtypes(include=['integer']):
            df[col] = pd.to_numeric(df[col], downcast='integer')
        return df

    print("Loading Banksim...")
    bs = pd.read_csv(os.path.join(banksim_path, 'bs140513_032310.csv.gz'))
    bs.columns = bs.columns.str.replace("'", "")
    for col in ['customer', 'category']:
        bs[col] = bs[col].str.replace("'", "").astype('category')

    start_date = datetime(year=2024, month=11, day=1)
    bs_master = pd.DataFrame({
        'transaction_id': np.arange(len(bs), dtype='int32'),
        'customer_id': bs['customer'],
        'amount_cad': (bs['amount'] * 1.62).astype('float32'),
        'debit_credit': 'debit',
        'transaction_datetime': bs['step'].apply(lambda x: start_date + timedelta(days=x)),
        'merchant_category': bs['category'],
        'ecommerce_ind': np.where(bs['category'].isin(['es_tech', 'es_contents']), 1, 0).astype('int8'),
        'cash_indicator': np.int8(0),
        'country': 'Canada', 'province': 'Ontario', 'city': 'Toronto',
        'source_dataset': 'banksim'
    })
    del bs
    gc.collect()

    def process_scotia(name, filename):
        print(f"  Processing {name}...")
        df = pd.read_csv(os.path.join(data_path, filename))
        df['customer_id'] = df['customer_id'].astype('category')
        df['source_dataset'] = name
        return shrink_and_categorize(df)

    card_master = process_scotia('scotia_card', 'card.csv.gz')
    card_master['cash_indicator'] = np.int8(0)

    abm_master = process_scotia('scotia_abm', 'abm.csv.gz')
    abm_master['merchant_category'] = 'ATM_WITHDRAWAL'
    abm_master['ecommerce_ind'] = np.int8(0)
    abm_master['cash_indicator'] = np.int8(1)

    others_list = []
    for name in ['cheque', 'eft', 'emt', 'westernunion', 'wire']:
        tmp = process_scotia(f'scotia_{name}', f'{name}.csv.gz')
        tmp['merchant_category'] = name.upper()
        others_list.append(tmp)

    others = pd.concat(others_list, axis=0)
    others['country'], others['province'], others['city'] = 'Canada', 'N/A', 'N/A'
    others['ecommerce_ind'] = np.int8(0)
    others['cash_indicator'] = np.int8(0)
    del others_list
    gc.collect()

    print("Final merge...")
    master_pool = pd.concat([bs_master, card_master, abm_master, others], axis=0, ignore_index=True)
    del bs_master, card_master, abm_master, others
    gc.collect()

    master_pool['city'] = master_pool['city'].fillna('UNKNOWN').astype('category')
    master_pool['transaction_datetime'] = pd.to_datetime(master_pool['transaction_datetime'])
    master_pool = compute_temporal_customer_features(master_pool)
    return shrink_and_categorize(master_pool)


def get_maps(master_pool):
    cust_map, num_cust = get_mapping(master_pool['customer_id'])
    cat_map, num_cat = get_mapping(master_pool['merchant_category'])
    city_map, num_city = get_mapping(master_pool['city'].str.upper().fillna('UNKNOWN'))
    master_pool['cust_idx'] = master_pool['customer_id'].map(cust_map)
    master_pool['cat_idx'] = master_pool['merchant_category'].map(cat_map)
    master_pool['city_idx'] = master_pool['city'].str.upper().fillna('UNKNOWN').map(city_map)
    return cust_map, cat_map, city_map, num_cust, num_cat, num_city


def master_pool_cache_is_current(path):
    if not os.path.exists(path):
        return False

    cached_cols = set(pd.read_csv(path, compression='gzip', nrows=0).columns)
    return MASTER_POOL_REQUIRED_COLS.issubset(cached_cols) and 'is_fraud' not in cached_cols


master_path = "master_transaction_pool.csv.gz"
if not master_pool_cache_is_current(master_path):
    print("Building master transaction pool")
    master_pool = build_master_transaction_pool(banksim_path, data_path)
    cust_map, cat_map, city_map, num_cust, num_cat, num_city = get_maps(master_pool)
    master_pool.to_csv(master_path, index=False, compression="gzip")
else:
    print("Reading master transaction pool from cache")
    master_pool = pd.read_csv(master_path, compression="gzip")
    cust_map, cat_map, city_map, num_cust, num_cat, num_city = get_maps(master_pool)

print(f"\nCustomers: {num_cust:,} | Categories: {num_cat} | Cities: {num_city}")
print(f"Transactions: {len(master_pool):,}")
master_pool.head(3)


# Prepare customer nodes for Graph Attention Network

We group individual customers and small businesses into one node type for simplicity. 

**Schema:** customer_id, age, income, tenure, sales, emp_count, is_biz, fraud_label

**Field definitions**
- **customer_id**: Unique customer identifier across KYC and BankSim sources.
- **age**: Customer age (years); derived from KYC birth dates or BankSim age buckets, with median imputation for missing.
- **income**: Annual income (or reported income proxy); missing values filled with the KYC median.
- **tenure**: Customer tenure in days since onboarding; defaults to 730 days if unknown.
- **sales**: Annual sales for small businesses; 0 for individuals, median-imputed when missing.
- **emp_count**: Number of employees for small businesses; 0 for individuals, median-imputed when missing.
- **is_biz**: 1 for small business customers, 0 for individuals.
- **fraud_label**: Customer-node label. BankSim customers are marked fraudulent if they have at least one fraudulent transaction; Scotiabank uses the provided customer labels; unlabeled customers are -1.


In [ ]:
def compute_tenure_days(df, date_col='onboard_date'):
    if date_col in df.columns:
        return (pd.Timestamp('2025-01-31') - pd.to_datetime(df[date_col], errors='coerce')).dt.days
    return pd.Series([np.nan] * len(df))


CUSTOMER_POOL_REQUIRED_COLS = {
    'customer_id', 'age', 'income', 'tenure', 'sales',
    'emp_count', 'is_biz', 'cust_idx', 'fraud_label'
}


def build_customer_label_lookup():
    scotia_labels = pd.read_csv(os.path.join(data_path, 'labels.csv.gz'))
    scotia_labels['customer_id'] = scotia_labels['customer_id'].astype(str)
    scotia_labels = scotia_labels.rename(columns={'label': 'fraud_label'})
    scotia_labels['fraud_label'] = scotia_labels['fraud_label'].astype('int8')

    banksim_labels = pd.read_csv(
        os.path.join(banksim_path, 'bs140513_032310.csv.gz'),
        usecols=['customer', 'fraud']
    )
    banksim_labels['customer_id'] = banksim_labels['customer'].astype(str).str.replace("'", '', regex=False)
    banksim_customer_labels = (
        banksim_labels.groupby('customer_id')['fraud']
        .max()
        .astype('int8')
        .rename('fraud_label')
        .reset_index()
    )

    combined_labels = pd.concat(
        [scotia_labels[['customer_id', 'fraud_label']], banksim_customer_labels],
        axis=0,
        ignore_index=True
    )
    return combined_labels.groupby('customer_id')['fraud_label'].max()


def build_customer_pool():
    label_lookup = build_customer_label_lookup()

    kyc_ind = pd.read_csv(os.path.join(data_path, 'kyc_individual.csv.gz'))
    kyc_biz = pd.read_csv(os.path.join(data_path, 'kyc_smallbusiness.csv.gz'))
    kyc_ind.columns = kyc_ind.columns.str.lower()
    kyc_biz.columns = kyc_biz.columns.str.lower()

    if 'birth_date' in kyc_ind.columns:
        birth_dates = pd.to_datetime(kyc_ind['birth_date'], errors='coerce')
        derived_age = (pd.Timestamp('2025-01-31') - birth_dates).dt.days / 365.25
        ind_age = pd.to_numeric(kyc_ind.get('age', pd.Series(dtype=float)), errors='coerce').fillna(derived_age)
    else:
        ind_age = pd.to_numeric(kyc_ind.get('age', pd.Series(dtype=float)), errors='coerce')

    ind = pd.DataFrame({
        'customer_id': kyc_ind['customer_id'].astype(str),
        'age': ind_age,
        'income': kyc_ind.get('income', np.nan),
        'tenure': compute_tenure_days(kyc_ind),
        'sales': 0.0, 'emp_count': 0.0, 'is_biz': 0
    })
    biz = pd.DataFrame({
        'customer_id': kyc_biz['customer_id'].astype(str),
        'age': np.nan,
        'income': kyc_biz.get('income', np.nan),
        'tenure': compute_tenure_days(kyc_biz),
        'sales': kyc_biz.get('sales', 0.0),
        'emp_count': kyc_biz.get('emp_count', 0.0),
        'is_biz': 1
    })

    median_income = ind['income'].median()
    median_age = ind['age'].median() if not pd.isna(ind['age'].median()) else 40.0
    median_sales = biz['sales'].median() if hasattr(biz['sales'], 'median') else 0.0
    median_emp = biz['emp_count'].median() if hasattr(biz['emp_count'], 'median') else 0.0

    for df in (ind, biz):
        df['income'] = df['income'].fillna(median_income)
        df['age'] = df['age'].fillna(median_age)
        df['tenure'] = df['tenure'].fillna(730)
        df['sales'] = df['sales'].fillna(median_sales)
        df['emp_count'] = df['emp_count'].fillna(median_emp)

    banksim = pd.read_csv(os.path.join(banksim_path, 'bs140513_032310.csv.gz'), usecols=['customer', 'age'])
    banksim['customer'] = banksim['customer'].astype(str).str.replace("'", '', regex=False)
    banksim['age'] = banksim['age'].astype(str).str.strip("'")
    age_map_bs = {'0': 18, '1': 22, '2': 30, '3': 40, '4': 50, '5': 60, '6': 70, 'U': median_age}
    banksim['age_num'] = banksim['age'].map(age_map_bs).fillna(median_age)

    banksim_customers = master_pool.loc[master_pool['source_dataset'] == 'banksim', 'customer_id'].astype(str).unique()
    banksim_age_lkp = banksim.drop_duplicates('customer').set_index('customer')['age_num']
    banksim_df = pd.DataFrame({
        'customer_id': banksim_customers,
        'age': [banksim_age_lkp.get(c, median_age) for c in banksim_customers],
        'income': median_income, 'tenure': 730.0, 'sales': 0.0, 'emp_count': 0.0, 'is_biz': 0
    })

    customers_df = pd.concat([ind, biz, banksim_df], axis=0, ignore_index=True)
    customers_df['cust_idx'] = customers_df['customer_id'].map(cust_map)
    customers_df = customers_df.dropna(subset=['cust_idx']).sort_values('cust_idx').reset_index(drop=True)
    customers_df[['age','income','tenure','sales','emp_count','is_biz']] =         customers_df[['age','income','tenure','sales','emp_count','is_biz']].astype(float)
    customers_df['cust_idx'] = customers_df['cust_idx'].astype(int)
    customers_df['fraud_label'] = customers_df['customer_id'].map(label_lookup).fillna(-1).astype('int8')
    return customers_df


def customer_pool_cache_is_current(path):
    if not os.path.exists(path):
        return False

    cached_cols = set(pd.read_csv(path, compression='gzip', nrows=0).columns)
    return CUSTOMER_POOL_REQUIRED_COLS.issubset(cached_cols)


cust_path = "customer_pool.csv.gz"
if not customer_pool_cache_is_current(cust_path):
    print("Building customer pool")
    customer_pool = build_customer_pool()
    customer_pool.to_csv(cust_path, index=False, compression="gzip")
else:
    customer_pool = pd.read_csv(cust_path, compression="gzip")

print(f"Customer pool: {len(customer_pool):,} rows")
customer_pool.head(3)


# Build Inductive Node Features

**Key design decision:** Instead of passing transaction features as edge attributes (which GAT does), we *pre-aggregate* them into the **customer node vector itself**. This is what makes GraphSAGE inductive:

- A new customer only needs their own KYC + transaction aggregates computed — no graph rebuild
- Temporal transaction signals are computed in timestamp order, so each transaction-derived feature only uses that customer’s prior history
- The model generalises from the *structure of the neighborhood* rather than memorised node IDs

**Customer node (13 features):** KYC (6) + transaction aggregates (7)  
**Category / City nodes:** learned embeddings (one-hot init)


In [ ]:
import torch
import numpy as np
from sklearn.preprocessing import StandardScaler


KYC_FEATURE_COLS = [
    'age', 'income', 'tenure', 'sales', 'emp_count', 'is_biz',
]
TRANSACTION_FEATURE_COLS = [
    'avg_txn_amount', 'max_txn_amount', 'std_txn_amount', 'txn_count',
    'cash_rate', 'ecom_rate', 'avg_24h_velocity', 'unique_cities',
    'unique_categories', 'min_time_delta', 'time_span_hours', 'geo_velocity',
]
CUSTOMER_FEATURE_COLS = KYC_FEATURE_COLS + TRANSACTION_FEATURE_COLS


def build_customer_feature_frame(customer_pool, master_pool):
    master_pool = master_pool.copy()
    master_pool['transaction_datetime'] = pd.to_datetime(
        master_pool['transaction_datetime'], errors='coerce'
    )

    # ── Leakage-safe transaction aggregates per customer ─────────────────────
    txn_agg = master_pool.groupby('cust_idx').agg(
        avg_txn_amount   = ('amount_cad', 'mean'),
        max_txn_amount   = ('amount_cad', 'max'),
        std_txn_amount   = ('amount_cad', 'std'),
        txn_count        = ('amount_cad', 'count'),
        cash_rate        = ('cash_indicator', 'mean'),
        ecom_rate        = ('ecommerce_ind', 'mean'),
        avg_24h_velocity = ('velocity_24h', 'mean'),
        unique_cities    = ('city', 'nunique'),
        unique_categories= ('merchant_category', 'nunique'),
        first_txn_at     = ('transaction_datetime', 'min'),
        last_txn_at      = ('transaction_datetime', 'max'),
    ).reset_index()

    min_time_delta = (
        master_pool.assign(
            time_delta_nonzero=master_pool['time_delta'].where(master_pool['time_delta'] > 0)
        )
        .groupby('cust_idx')['time_delta_nonzero']
        .min()
        .rename('min_time_delta')
        .reset_index()
    )
    txn_agg = txn_agg.merge(min_time_delta, on='cust_idx', how='left')
    txn_agg['time_span_hours'] = (
        (txn_agg['last_txn_at'] - txn_agg['first_txn_at'])
        .dt.total_seconds()
        .div(3600)
        .fillna(0.0)
    )

    geo_denominator = np.maximum(
        txn_agg['time_span_hours'].to_numpy(dtype=np.float32),
        1.0 / 60.0,
    )
    unique_city_counts = txn_agg['unique_cities'].to_numpy(dtype=np.float32)
    txn_agg['geo_velocity'] = np.where(
        unique_city_counts > 1,
        unique_city_counts / geo_denominator,
        0.0,
    )
    txn_agg = txn_agg.drop(columns=['first_txn_at', 'last_txn_at'])

    # ── Merge with customer KYC + customer-node labels ──────────────────────
    cp = customer_pool.copy()
    cp['cust_idx'] = cp['cust_idx'].astype(int)
    cp = cp.sort_values('cust_idx').reset_index(drop=True)
    cp = cp.merge(txn_agg, on='cust_idx', how='left')

    for col in TRANSACTION_FEATURE_COLS:
        cp[col] = cp[col].fillna(0.0)

    cp['unique_cities'] = cp['unique_cities'].clip(lower=0)
    cp['unique_categories'] = cp['unique_categories'].clip(lower=0)
    cp['min_time_delta'] = cp['min_time_delta'].clip(lower=0)
    cp['time_span_hours'] = cp['time_span_hours'].clip(lower=0)
    cp['geo_velocity'] = cp['geo_velocity'].clip(lower=0)

    expected_cust_idx = np.arange(num_cust)
    actual_cust_idx = cp['cust_idx'].to_numpy(dtype=int)
    if not np.array_equal(actual_cust_idx, expected_cust_idx):
        missing = np.setdiff1d(expected_cust_idx, actual_cust_idx)
        raise ValueError(f"Customer feature frame is missing cust_idx values: {missing[:10]}")

    missing_feature_cols = [col for col in CUSTOMER_FEATURE_COLS if col not in cp.columns]
    if missing_feature_cols:
        raise ValueError(f"Customer feature frame is missing columns: {missing_feature_cols}")

    return cp


cp = build_customer_feature_frame(customer_pool, master_pool)
customer_labels = cp['fraud_label'].to_numpy(dtype=np.int8)

# Scale customer features
cust_raw = cp[CUSTOMER_FEATURE_COLS].values.astype(np.float32)
cust_scaler = StandardScaler()
cust_scaled = cust_scaler.fit_transform(cust_raw)
cust_scaled = np.nan_to_num(cust_scaled, nan=0.0, posinf=3.0, neginf=-3.0)
x_cust = torch.tensor(cust_scaled, dtype=torch.float)

# Category / city embeddings (one-hot init — model will learn better representations)
x_cat  = torch.eye(num_cat)
x_city = torch.eye(num_city)

print(f"Customer node matrix : {x_cust.shape}  ({len(CUSTOMER_FEATURE_COLS)} features)")
print(f"Feature columns: {CUSTOMER_FEATURE_COLS}")


# Define Node & Edge Features

Node features for each entity type and graph edges:
- **Customer nodes**: age, income, tenure, sales, emp_count, is_biz, plus aggregated transaction behaviour (`avg_txn_amount`, `max_txn_amount`, `std_txn_amount`, `txn_count`, `cash_rate`, `ecom_rate`, `avg_24h_velocity`)
- **Category nodes**: one-hot identity matrix
- **City nodes**: one-hot identity matrix
- **Edges**: topology only (`customer → category`, `customer → city`); fraud labels and temporal transaction statistics are attached to customer nodes, not transaction edges

The temporal aggregate `avg_24h_velocity` is built from a prior-only rolling 24h transaction count, so downstream node features stay leakage-safe.


In [ ]:
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T

# ── Build heterogeneous graph ─────────────────────────────────────────────────
data_sage = HeteroData()
data_sage['customer'].x = x_cust
data_sage['category'].x = x_cat
data_sage['city'].x     = x_city

# Edges: customer → category  (each transaction row is one edge)
data_sage['customer', 'purchases_at', 'category'].edge_index = torch.stack([
    torch.tensor(master_pool['cust_idx'].values, dtype=torch.long),
    torch.tensor(master_pool['cat_idx'].values,  dtype=torch.long)
])

# Edges: customer → city
data_sage['customer', 'transacts_in', 'city'].edge_index = torch.stack([
    torch.tensor(master_pool['cust_idx'].values,  dtype=torch.long),
    torch.tensor(master_pool['city_idx'].values,  dtype=torch.long)
])

# Add reverse edges so messages flow back to customers (hub → customer)
data_sage = T.ToUndirected()(data_sage)

# ── Customer-node labels & masks ─────────────────────────────────────────────
if len(customer_labels) != num_cust:
    raise ValueError(f"Expected {num_cust} customer labels, found {len(customer_labels)}")

num_customer_nodes = data_sage['customer'].num_nodes
if num_customer_nodes != num_cust:
    raise ValueError(f"Expected {num_cust} customer nodes, found {num_customer_nodes}")

data_sage['customer'].y = torch.tensor(customer_labels, dtype=torch.float)

labeled_mask        = customer_labels != -1
fraud_customers     = (customer_labels == 1).sum()
legit_customers     = (customer_labels == 0).sum()
unlabeled_customers = (customer_labels == -1).sum()
print(f"Labels: {fraud_customers:,} fraud | {legit_customers:,} legit | {unlabeled_customers:,} unlabeled")
print(f"Fraud rate (labeled): {fraud_customers / (fraud_customers + legit_customers) * 100:.2f}%")

# Train / val / test split (80 / 10 / 10) across customer nodes
num_customers = num_customer_nodes
perm = torch.randperm(num_customers)
train_size = int(0.8 * num_customers)
val_size   = int(0.1 * num_customers)

train_mask = torch.zeros(num_customers, dtype=torch.bool)
val_mask   = torch.zeros(num_customers, dtype=torch.bool)
test_mask  = torch.zeros(num_customers, dtype=torch.bool)
train_mask[perm[:train_size]] = True
val_mask[perm[train_size:train_size + val_size]] = True
test_mask[perm[train_size + val_size:]] = True

if (train_mask & val_mask).any() or (train_mask & test_mask).any() or (val_mask & test_mask).any():
    raise ValueError('Customer-node splits overlap; expected disjoint train/val/test masks')

data_sage['customer'].train_mask = train_mask
data_sage['customer'].val_mask   = val_mask
data_sage['customer'].test_mask  = test_mask

print(f"\nGraph: {data_sage.num_nodes:,} nodes | {data_sage.num_edges:,} edges")
print("Customer-node split summary (total / labeled / unlabeled):")
for split_name, split_mask in {
    'Train': train_mask,
    'Val': val_mask,
    'Test': test_mask,
}.items():
    if split_mask.numel() != num_customer_nodes:
        raise ValueError(f"{split_name} mask has {split_mask.numel()} entries, expected {num_customer_nodes}")
    split_labels = customer_labels[split_mask.cpu().numpy()]
    split_labeled = split_labels != -1
    print(
        f"  {split_name:<5} {int(split_mask.sum()):,} total | "
        f"{int(split_labeled.sum()):,} labeled | {int((~split_labeled).sum()):,} unlabeled"
    )
for et in data_sage.edge_types:
    print(f"  {et}: {data_sage[et].num_edges:,} edges")


# Define Neighbor Loader

Since there are almost 26M edges, it is not possible to load the entire graph at once. Therefore, during training and inference we only look at nodes which are at most 2-hops away. This makes sense since fraud networks are usually isolated to pockets of criminals.

In [9]:
from torch_geometric.loader import NeighborLoader

BATCH_SIZE   = 512
# 2-hop sampling: sample up to 15 1-hop and 10 2-hop neighbors per node
NUM_NEIGHBORS = [15, 10]

train_loader = NeighborLoader(
    data_sage,
    num_neighbors=NUM_NEIGHBORS,
    batch_size=BATCH_SIZE,
    input_nodes=('customer', data_sage['customer'].train_mask),
    shuffle=True,
    num_workers=0,
)
val_loader = NeighborLoader(
    data_sage,
    num_neighbors=NUM_NEIGHBORS,
    batch_size=BATCH_SIZE,
    input_nodes=('customer', data_sage['customer'].val_mask),
    shuffle=False,
    num_workers=0,
)

print(f"Train: ~{len(train_loader)} batches | Val: ~{len(val_loader)} batches")
print(f"Batch size: {BATCH_SIZE} | Hops: {NUM_NEIGHBORS}")


/student/minalex/.local/lib/python3.10/site-packages/torch_geometric/loader/neighbor_loader.py:229: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(


Train: ~103 batches | Val: ~13 batches
Batch size: 512 | Hops: [15, 10]


# GraphSAGE Model Definition

**Architecture:** 3-layer heterogeneous GraphSAGE

- **Layer 1:** Category/City → Customer (aggregate hub signals into customer embedding)
- **Layer 2:** Customer → Category/City (update hub embeddings with customer context)
- **Layer 3:** Category/City → Customer (final aggregation back to customer)
- **Residual connection:** input projection added to Layer-1 output for stable gradient flow
- **Loss:** Focal Loss (same as GAT) to handle extreme class imbalance

**Why SAGEConv?**  
SAGEConv learns `f(h_v, MEAN(h_u for u in N(v)))` — a *function* of features, not an ID lookup. The exact same weights score any new node.


In [12]:
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, HeteroConv, Linear

# Suppress expected "node type not updated" warnings — each HeteroConv layer
# intentionally updates only a subset of node types (hub→customer or customer→hub).
# PyG warns per-layer without knowing the full 3-layer pipeline context.
warnings.filterwarnings('ignore', message='.*do not occur as destination type.*')


def focal_loss(y_pred, y_true, alpha=0.25, gamma=2.0):
    """Focal Loss for extreme imbalance. Ignores -1 (unlabeled) nodes."""
    mask = y_true != -1
    if mask.sum() == 0:
        return torch.tensor(0.0, requires_grad=True).to(y_pred.device)
    logits  = y_pred[mask]
    targets = y_true[mask].float()
    probs   = torch.sigmoid(logits)
    bce     = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    p_t     = targets * probs + (1 - targets) * (1 - probs)
    loss    = bce * ((1 - p_t) ** gamma)
    if alpha >= 0:
        alpha_t = targets * alpha + (1 - targets) * (1 - alpha)
        loss = alpha_t * loss
    return loss.mean()


class FraudSAGE(torch.nn.Module):
    """
    Heterogeneous GraphSAGE for inductive fraud detection.

    3-layer message passing:
      conv1: Hub → Customer  (aggregate category/city signals)
      conv2: Customer → Hub  (update hub embeddings)
      conv3: Hub → Customer  (final customer embedding)

    A residual connection between the input projection and conv1 output
    stabilises training and helps gradient flow on deep hetero graphs.
    """

    def __init__(self, hidden_channels: int):
        super().__init__()

        # Layer 1: aggregate from hubs into customer nodes
        self.conv1 = HeteroConv({
            ('category', 'rev_purchases_at', 'customer'): SAGEConv((-1, -1), hidden_channels, aggr='mean'),
            ('city',     'rev_transacts_in',  'customer'): SAGEConv((-1, -1), hidden_channels, aggr='mean'),
        }, aggr='sum')

        self.bn1 = nn.BatchNorm1d(hidden_channels)

        # Layer 2: propagate customer context back to hubs
        self.conv2 = HeteroConv({
            ('customer', 'purchases_at', 'category'): SAGEConv((-1, -1), hidden_channels, aggr='mean'),
            ('customer', 'transacts_in', 'city'):     SAGEConv((-1, -1), hidden_channels, aggr='mean'),
        }, aggr='sum')

        # Layer 3: final aggregation back to customers
        self.conv3 = HeteroConv({
            ('category', 'rev_purchases_at', 'customer'): SAGEConv((-1, -1), hidden_channels, aggr='mean'),
            ('city',     'rev_transacts_in',  'customer'): SAGEConv((-1, -1), hidden_channels, aggr='mean'),
        }, aggr='sum')

        self.bn2 = nn.BatchNorm1d(hidden_channels)

        # Residual projection from raw customer features → hidden
        self.input_proj = Linear(-1, hidden_channels)

        # Final classifier head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ELU(),
            nn.Dropout(p=0.2),
            nn.Linear(hidden_channels // 2, 1),
        )

    def forward(self, x_dict, edge_index_dict):
        # Residual shortcut from raw input
        res = F.elu(self.input_proj(x_dict['customer']))

        # Layer 1: Hub → Customer
        h1 = self.conv1(x_dict, edge_index_dict)
        h1['customer'] = F.elu(self.bn1(h1['customer']) + res)
        # Keep original hub features for conv2
        for k in x_dict:
            if k not in h1:
                h1[k] = x_dict[k]

        # Layer 2: Customer → Hub
        h2 = self.conv2(h1, edge_index_dict)
        h2 = {k: F.elu(v) for k, v in h2.items()}
        if 'customer' not in h2:
            h2['customer'] = h1['customer']

        # Layer 3: Hub → Customer (final)
        h3 = self.conv3(h2, edge_index_dict)
        h3['customer'] = F.elu(self.bn2(h3['customer']) + h1['customer'])

        return self.classifier(h3['customer'])   # (N, 1)


HIDDEN_CHANNELS = 64
model_sage = FraudSAGE(hidden_channels=HIDDEN_CHANNELS)
print(f"FraudSAGE initialized — hidden_channels={HIDDEN_CHANNELS}")

# SAGEConv / Linear with in_channels=-1 are lazy modules: parameters are
# allocated on the first forward pass.  Run one dummy batch to materialise them.
_dummy = next(iter(train_loader))
with torch.no_grad():
    model_sage(_dummy.x_dict, _dummy.edge_index_dict)

total_params = sum(p.numel() for p in model_sage.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")


FraudSAGE initialized — hidden_channels=64


RuntimeError: Numpy is not available

In [ ]:
from tqdm import tqdm
import gc

# ── Hyperparameters ───────────────────────────────────────────────────────────
LEARNING_RATE   = 0.001
WEIGHT_DECAY    = 1e-5
NUM_EPOCHS      = 100
FOCAL_ALPHA     = 0.50
FOCAL_GAMMA     = 1.5

# Pseudo-labeling
ENABLE_PL       = True
FRAUD_THRESHOLD = 0.95
LEGIT_THRESHOLD = 0.05
WARMUP_EPOCHS   = 20

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def make_customer_loader(graph, mask, batch_size, shuffle):
    return NeighborLoader(
        graph,
        num_neighbors=NUM_NEIGHBORS,
        batch_size=batch_size,
        input_nodes=('customer', mask),
        shuffle=shuffle,
        num_workers=0,
    )


def collect_all_customer_probs(model, graph, batch_size_multiplier=2):
    model.eval()
    all_customer_node_probs = torch.zeros(graph['customer'].num_nodes)
    loader = NeighborLoader(
        graph,
        num_neighbors=NUM_NEIGHBORS,
        batch_size=BATCH_SIZE * batch_size_multiplier,
        input_nodes=('customer', torch.ones(graph['customer'].num_nodes, dtype=torch.bool)),
        shuffle=False,
        num_workers=0,
    )
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch.x_dict, batch.edge_index_dict)
            batch_size_local = batch['customer'].batch_size
            probs = torch.sigmoid(logits.squeeze()[:batch_size_local]).cpu()
            all_customer_node_probs[batch['customer'].n_id[:batch_size_local]] = probs
    del loader
    gc.collect()
    return all_customer_node_probs


def collect_customer_predictions(model, graph, mask, batch_size_multiplier=2, return_logits=False):
    model.eval()
    loader = make_customer_loader(
        graph,
        mask,
        batch_size=BATCH_SIZE * batch_size_multiplier,
        shuffle=False,
    )
    all_probs, all_labels, all_preds, all_node_ids, all_logits = [], [], [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch.x_dict, batch.edge_index_dict).squeeze()
            batch_size_local = batch['customer'].batch_size
            logits = logits[:batch_size_local].detach().cpu()
            probs = torch.sigmoid(logits)
            labels = batch['customer'].y[:batch_size_local].cpu()
            node_ids = batch['customer'].n_id[:batch_size_local].cpu()
            labeled_mask = labels != -1
            if labeled_mask.sum() == 0:
                continue
            all_probs.extend(probs[labeled_mask].tolist())
            all_labels.extend(labels[labeled_mask].tolist())
            all_preds.extend((probs[labeled_mask] > 0.5).float().tolist())
            all_node_ids.extend(node_ids[labeled_mask].tolist())
            if return_logits:
                all_logits.extend(logits[labeled_mask].tolist())
    del loader
    gc.collect()
    result = {
        'probs': np.asarray(all_probs, dtype=np.float32),
        'labels': np.asarray(all_labels, dtype=np.int64),
        'preds': np.asarray(all_preds, dtype=np.int64),
        'node_ids': np.asarray(all_node_ids, dtype=np.int64),
    }
    if return_logits:
        result['logits'] = np.asarray(all_logits, dtype=np.float32)
    return result


def summarize_confidence_profile(probs):
    probs = np.asarray(probs, dtype=np.float64)
    probs = np.clip(probs, 1e-6, 1 - 1e-6)
    margins = np.abs(probs - 0.5)
    entropy = -(probs * np.log(probs) + (1 - probs) * np.log(1 - probs))
    return {
        'mean_confidence_margin': float(margins.mean()),
        'median_confidence_margin': float(np.median(margins)),
        'uncertain_share_40_60': float(((probs >= 0.4) & (probs <= 0.6)).mean()),
        'confident_share_10_90': float(((probs <= 0.1) | (probs >= 0.9)).mean()),
        'mean_entropy': float(entropy.mean()),
        'q10_prob': float(np.quantile(probs, 0.10)),
        'q50_prob': float(np.quantile(probs, 0.50)),
        'q90_prob': float(np.quantile(probs, 0.90)),
    }


def capture_confidence_round(model, graph, mask, epoch, round_idx, split_name='validation'):
    prediction_snapshot = collect_customer_predictions(model, graph, mask)
    summary = summarize_confidence_profile(prediction_snapshot['probs'])
    summary.update({
        'round_idx': int(round_idx),
        'epoch': int(epoch),
        'split': split_name,
        'num_scored_nodes': int(len(prediction_snapshot['probs'])),
    })
    return summary


def train_fraud_sage_model(base_graph, feature_names, num_epochs=NUM_EPOCHS, enable_pl=ENABLE_PL,
                           verbose=True, run_label='Main run'):
    run_graph = base_graph.clone()
    train_loader_graph = make_customer_loader(
        run_graph,
        run_graph['customer'].train_mask,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
    val_loader_graph = make_customer_loader(
        run_graph,
        run_graph['customer'].val_mask,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    model = FraudSAGE(hidden_channels=HIDDEN_CHANNELS).to(device)
    optimizer_local = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    dummy_batch = next(iter(train_loader_graph))
    with torch.no_grad():
        model(dummy_batch.x_dict, dummy_batch.edge_index_dict)

    if verbose:
        total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        train_labels_init = run_graph['customer'].y[run_graph['customer'].train_mask]
        labeled_train = train_labels_init != -1
        print(f"\n[{run_label}] Feature set size: {len(feature_names)}")
        print(f"[{run_label}] Training customer nodes — fraud: {(train_labels_init[labeled_train] == 1).sum():,} | "
              f"legit: {(train_labels_init[labeled_train] == 0).sum():,} | "
              f"unlabeled: {(train_labels_init == -1).sum():,}")
        print(f"[{run_label}] Device: {device} | Epochs: {num_epochs} | Trainable parameters: {total_params:,}")

    total_pseudo_labeled = 0
    pseudo_label_history = []
    confidence_round_history = [
        capture_confidence_round(
            model,
            run_graph,
            run_graph['customer'].val_mask,
            epoch=-1,
            round_idx=0,
            split_name='validation',
        )
    ]

    epoch_iter = tqdm(range(num_epochs), desc=run_label, disable=not verbose)
    for epoch in epoch_iter:
        model.train()
        epoch_loss = 0.0
        num_batches = 0

        for batch in train_loader_graph:
            batch = batch.to(device)
            optimizer_local.zero_grad()
            logits = model(batch.x_dict, batch.edge_index_dict).squeeze()
            batch_size_local = batch['customer'].batch_size
            logits = logits[:batch_size_local]
            labels = batch['customer'].y[:batch_size_local]
            labeled_mask = labels != -1
            if labeled_mask.sum() == 0:
                continue
            loss = focal_loss(logits[labeled_mask], labels[labeled_mask], alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
            loss.backward()
            optimizer_local.step()
            epoch_loss += loss.item()
            num_batches += 1

        avg_loss = epoch_loss / num_batches if num_batches > 0 else 0.0

        if enable_pl and epoch >= WARMUP_EPOCHS and epoch % 5 == 0:
            all_customer_node_probs = collect_all_customer_probs(model, run_graph)
            pseudo_candidate_mask = (run_graph['customer'].y == -1) & run_graph['customer'].train_mask
            high_conf_fraud = (all_customer_node_probs > FRAUD_THRESHOLD) & pseudo_candidate_mask
            high_conf_legit = (all_customer_node_probs < LEGIT_THRESHOLD) & pseudo_candidate_mask
            nf = int(high_conf_fraud.sum().item())
            nl = int(high_conf_legit.sum().item())
            if nf > 0 or nl > 0:
                run_graph['customer'].y[high_conf_fraud] = 1.0
                run_graph['customer'].y[high_conf_legit] = 0.0
                total_pseudo_labeled += nf + nl
            pseudo_label_history.append({
                'round_idx': len(pseudo_label_history) + 1,
                'epoch': int(epoch),
                'customer_fraud': nf,
                'customer_legit': nl,
                'customer_total': int(total_pseudo_labeled),
            })
            confidence_round_history.append(
                capture_confidence_round(
                    model,
                    run_graph,
                    run_graph['customer'].val_mask,
                    epoch=epoch,
                    round_idx=len(confidence_round_history),
                    split_name='validation',
                )
            )
            if verbose and (nf > 0 or nl > 0):
                tqdm.write(
                    f"  → Pseudo-labeled training customer nodes: {nf} fraud, {nl} legit "
                    f"(total: {total_pseudo_labeled})"
                )

        if epoch % 5 == 0:
            val_snapshot = collect_customer_predictions(model, run_graph, run_graph['customer'].val_mask)
            val_acc = (val_snapshot['preds'] == val_snapshot['labels']).mean() if len(val_snapshot['labels']) else 0.0
            if verbose:
                tqdm.write(
                    f"Epoch {epoch:3d}: loss={avg_loss:.4f}  val_customer_acc={val_acc:.4f} "
                    f"({len(val_snapshot['labels']):,} labeled customer nodes)"
                )

    if verbose:
        print(f"\n[{run_label}] Training complete. Total pseudo-labeled training customer nodes: {total_pseudo_labeled:,}")

    return {
        'model': model,
        'optimizer': optimizer_local,
        'graph': run_graph,
        'total_pseudo_labeled': total_pseudo_labeled,
        'pseudo_label_history': pseudo_label_history,
        'confidence_round_history': confidence_round_history,
        'feature_names': list(feature_names),
    }


training_run = train_fraud_sage_model(data_sage, feature_names=CUSTOMER_FEATURE_COLS)
model_sage = training_run['model']
optimizer = training_run['optimizer']
data_graph = training_run['graph']
total_pseudo_labeled = training_run['total_pseudo_labeled']
pseudo_label_history = training_run['pseudo_label_history']
confidence_round_history = training_run['confidence_round_history']


In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, auc,
)
import matplotlib.pyplot as plt


def compute_topk_lift_table(labels, probs, topk_fracs=(0.01, 0.02, 0.05, 0.10)):
    labels = np.asarray(labels, dtype=np.int64)
    probs = np.asarray(probs, dtype=np.float64)
    baseline_rate = labels.mean() if len(labels) else np.nan
    order = np.argsort(-probs)
    ranked_labels = labels[order]
    total_frauds = ranked_labels.sum()
    rows = []
    for frac in topk_fracs:
        k = max(1, int(np.ceil(len(ranked_labels) * frac)))
        top_labels = ranked_labels[:k]
        frauds_in_top_k = int(top_labels.sum())
        precision_at_k = frauds_in_top_k / k if k else np.nan
        recall_at_k = frauds_in_top_k / total_frauds if total_frauds else np.nan
        lift = precision_at_k / baseline_rate if baseline_rate and baseline_rate > 0 else np.nan
        rows.append({
            'top_pct': frac * 100,
            'customer_count': int(k),
            'frauds_captured': frauds_in_top_k,
            'baseline_fraud_rate': float(baseline_rate),
            'precision_at_k': float(precision_at_k),
            'recall_at_k': float(recall_at_k),
            'lift': float(lift),
        })
    return pd.DataFrame(rows)


test_snapshot = collect_customer_predictions(
    model_sage,
    data_graph,
    data_graph['customer'].test_mask,
    return_logits=True,
)
all_probs_test = test_snapshot['probs']
all_preds = test_snapshot['preds']
all_labels = test_snapshot['labels']
test_logits = test_snapshot['logits']

print(f"Test customer nodes: {len(all_labels):,} labeled ({int(all_labels.sum())} fraud, {int((all_labels == 0).sum())} legit)\n")
print(classification_report(all_labels, all_preds, target_names=['Legit', 'Fraud'], digits=4))

cm = confusion_matrix(all_labels, all_preds)
print(f"Confusion Matrix:\n  TN={cm[0,0]}  FP={cm[0,1]}\n  FN={cm[1,0]}  TP={cm[1,1]}")

prec_v, rec_v, _ = precision_recall_curve(all_labels, all_probs_test)
pr_auc = auc(rec_v, prec_v)
fr = cm[1,1] / (cm[1,1] + cm[1,0]) if (cm[1,1] + cm[1,0]) > 0 else 0.0
fp2 = cm[1,1] / (cm[1,1] + cm[0,1]) if (cm[1,1] + cm[0,1]) > 0 else 0.0
ff1 = 2 * fr * fp2 / (fr + fp2) if (fr + fp2) > 0 else 0.0

topk_lift_df = compute_topk_lift_table(all_labels, all_probs_test)
primary_topk_lift = float(topk_lift_df.loc[np.isclose(topk_lift_df['top_pct'], 5.0), 'lift'].iloc[0])
confidence_round_df = pd.DataFrame(confidence_round_history)

print(f"\nPR-AUC={pr_auc:.4f}")
print(f"Fraud Recall={fr:.4f}  Fraud Precision={fp2:.4f}  F1={ff1:.4f}")
print("\nTop-K lift (primary ranking metric):")
print(
    topk_lift_df[['top_pct', 'customer_count', 'frauds_captured', 'precision_at_k', 'recall_at_k', 'lift']]
    .rename(columns={
        'top_pct': 'Top %',
        'customer_count': 'Customers',
        'frauds_captured': 'Frauds',
        'precision_at_k': 'Precision@K',
        'recall_at_k': 'Recall@K',
        'lift': 'Lift',
    })
    .to_string(index=False, float_format=lambda x: f'{x:.4f}')
)

if not confidence_round_df.empty:
    print("\nConfidence stability over pseudo-labeling rounds:")
    print(
        confidence_round_df[
            ['round_idx', 'epoch', 'mean_confidence_margin', 'uncertain_share_40_60', 'confident_share_10_90', 'mean_entropy']
        ].to_string(index=False, float_format=lambda x: f'{x:.4f}')
    )

print(f"\n{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<10} {'TP'}")
print('-' * 55)
for thr in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    pt = (all_probs_test > thr).astype(int)
    tp = ((pt == 1) & (all_labels == 1)).sum()
    fp_ = ((pt == 1) & (all_labels == 0)).sum()
    fn_ = ((pt == 0) & (all_labels == 1)).sum()
    pr_ = tp / (tp + fp_) if (tp + fp_) > 0 else 0.0
    rc_ = tp / (tp + fn_) if (tp + fn_) > 0 else 0.0
    f1_ = 2 * pr_ * rc_ / (pr_ + rc_) if (pr_ + rc_) > 0 else 0.0
    print(f"{thr:<12.2f} {pr_:<12.4f} {rc_:<12.4f} {f1_:<10.4f} {tp}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(rec_v, prec_v, lw=2, color='tab:blue', label=f'PR-AUC={pr_auc:.3f}')
axes[0].axhline(all_labels.mean(), color='grey', linestyle='--', alpha=0.6, label='Fraud prevalence')
axes[0].set(title='Precision-Recall Curve', xlabel='Recall', ylabel='Precision')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(topk_lift_df['top_pct'], topk_lift_df['lift'], marker='o', lw=2, color='tab:orange')
axes[1].axhline(1.0, color='grey', linestyle='--', alpha=0.6)
for _, row in topk_lift_df.iterrows():
    axes[1].annotate(f"{row['lift']:.1f}x", (row['top_pct'], row['lift']), textcoords='offset points', xytext=(0, 6), ha='center')
axes[1].set(title='Top-K Lift', xlabel='Top scored customers (%)', ylabel='Lift vs. baseline')
axes[1].grid(alpha=0.3)

if not confidence_round_df.empty:
    axes[2].plot(confidence_round_df['round_idx'], confidence_round_df['mean_confidence_margin'], marker='o', lw=2, label='Mean |p - 0.5|')
    axes[2].plot(confidence_round_df['round_idx'], confidence_round_df['confident_share_10_90'], marker='s', lw=2, label='Share outside [0.1, 0.9]')
    axes[2].plot(confidence_round_df['round_idx'], confidence_round_df['uncertain_share_40_60'], marker='^', lw=2, label='Share inside [0.4, 0.6]')
    axes[2].set_xticks(confidence_round_df['round_idx'])
axes[2].set(title='Confidence Stability Across Pseudo-label Rounds', xlabel='Pseudo-label round', ylabel='Share / margin')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('model_evaluation_sage.png', dpi=150, bbox_inches='tight')
print("\n✓ Plots saved to model_evaluation_sage.png")
plt.show()


## Leave-One-Out Feature Analysis

To quantify feature dependence, the notebook now retrains the fraud model once per
customer-node feature with that feature removed from the final input schema.

The output ranks features by how much **PR-AUC** and especially **Top-5% lift** fall
when the feature is excluded.


In [ ]:
def drop_customer_feature(graph, feature_idx):
    ablated_graph = graph.clone()
    keep_indices = [i for i in range(graph['customer'].x.shape[1]) if i != feature_idx]
    ablated_graph['customer'].x = graph['customer'].x[:, keep_indices]
    return ablated_graph


leave_one_out_rows = []
for feature_idx, feature_name in enumerate(CUSTOMER_FEATURE_COLS):
    print(f"Running leave-one-out analysis without feature: {feature_name}")
    ablated_graph = drop_customer_feature(data_sage, feature_idx)
    ablated_feature_names = [f for j, f in enumerate(CUSTOMER_FEATURE_COLS) if j != feature_idx]
    loo_run = train_fraud_sage_model(
        ablated_graph,
        feature_names=ablated_feature_names,
        num_epochs=NUM_EPOCHS,
        enable_pl=ENABLE_PL,
        verbose=False,
        run_label=f'Drop {feature_name}',
    )
    loo_snapshot = collect_customer_predictions(
        loo_run['model'],
        loo_run['graph'],
        loo_run['graph']['customer'].test_mask,
    )
    loo_prec, loo_rec, _ = precision_recall_curve(loo_snapshot['labels'], loo_snapshot['probs'])
    loo_pr_auc = auc(loo_rec, loo_prec)
    loo_topk_lift_df = compute_topk_lift_table(loo_snapshot['labels'], loo_snapshot['probs'])
    loo_top5_lift = float(loo_topk_lift_df.loc[np.isclose(loo_topk_lift_df['top_pct'], 5.0), 'lift'].iloc[0])
    loo_cm = confusion_matrix(loo_snapshot['labels'], loo_snapshot['preds'])
    loo_recall = loo_cm[1, 1] / (loo_cm[1, 1] + loo_cm[1, 0]) if (loo_cm[1, 1] + loo_cm[1, 0]) > 0 else 0.0
    leave_one_out_rows.append({
        'dropped_feature': feature_name,
        'remaining_feature_count': len(ablated_feature_names),
        'pr_auc': float(loo_pr_auc),
        'delta_pr_auc': float(loo_pr_auc - pr_auc),
        'top_5pct_lift': float(loo_top5_lift),
        'delta_top_5pct_lift': float(loo_top5_lift - primary_topk_lift),
        'fraud_recall': float(loo_recall),
        'pseudo_labeled_total': int(loo_run['total_pseudo_labeled']),
    })

leave_one_out_results = pd.DataFrame(leave_one_out_rows).sort_values(
    by=['delta_top_5pct_lift', 'delta_pr_auc']
).reset_index(drop=True)

print('\nLeave-one-out impact (most harmful feature removals first):')
print(
    leave_one_out_results[
        ['dropped_feature', 'delta_top_5pct_lift', 'delta_pr_auc', 'top_5pct_lift', 'pr_auc', 'fraud_recall']
    ].to_string(index=False, float_format=lambda x: f'{x:.4f}')
)

fig, axes = plt.subplots(1, 2, figsize=(16, max(6, 0.35 * len(leave_one_out_results))))
plot_df = leave_one_out_results.sort_values('delta_top_5pct_lift')
axes[0].barh(plot_df['dropped_feature'], plot_df['delta_top_5pct_lift'], color='tab:red', alpha=0.8)
axes[0].axvline(0.0, color='grey', linestyle='--', alpha=0.6)
axes[0].set(title='Leave-one-out impact on Top-5% lift', xlabel='Δ lift after dropping feature', ylabel='Dropped feature')
axes[0].grid(axis='x', alpha=0.3)

plot_df_pr = leave_one_out_results.sort_values('delta_pr_auc')
axes[1].barh(plot_df_pr['dropped_feature'], plot_df_pr['delta_pr_auc'], color='tab:blue', alpha=0.8)
axes[1].axvline(0.0, color='grey', linestyle='--', alpha=0.6)
axes[1].set(title='Leave-one-out impact on PR-AUC', xlabel='Δ PR-AUC after dropping feature', ylabel='Dropped feature')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('feature_leave_one_out_sage.png', dpi=150, bbox_inches='tight')
print('\n✓ Leave-one-out figure saved to feature_leave_one_out_sage.png')
plt.show()


# Temperature Scaling for Calibration

Ranking quality is now reported with **Top-K lift**, **PR-AUC**, and the pseudo-label
**confidence-stability** profile above.

This section keeps temperature scaling as an optional post-processing step for the
probability scale itself. It checks whether calibration smooths or sharpens scores while
preserving ranking-oriented performance.


In [ ]:
from scipy.optimize import minimize
from sklearn.metrics import log_loss
from sklearn.calibration import calibration_curve

# Get validation set logits (before sigmoid)
val_snapshot = collect_customer_predictions(
    model_sage,
    data_graph,
    data_graph['customer'].val_mask,
    return_logits=True,
)
val_logits = val_snapshot['logits']
val_labels_calib = val_snapshot['labels']

# Original (uncalibrated) predictions
val_probs_original = 1 / (1 + np.exp(-val_logits))


def temperature_objective(T):
    """Negative log-likelihood (lower is better calibration)."""
    T = max(T[0], 0.01)
    calibrated_probs = 1 / (1 + np.exp(-val_logits / T))
    calibrated_probs = np.clip(calibrated_probs, 1e-7, 1 - 1e-7)
    return log_loss(val_labels_calib, calibrated_probs)


result = minimize(temperature_objective, x0=[1.0], method='Nelder-Mead', options={'maxiter': 100, 'xatol': 1e-4})
optimal_temperature = max(result.x[0], 0.01)

print('=' * 80)
print('TEMPERATURE SCALING CALIBRATION')
print('=' * 80)
print(f'Optimal Temperature: {optimal_temperature:.4f}')
print(f'Original NLL (T=1): {temperature_objective([1.0]):.4f}')
print(f'Calibrated NLL (T={optimal_temperature:.2f}): {temperature_objective([optimal_temperature]):.4f}')

val_probs_calibrated = 1 / (1 + np.exp(-val_logits / optimal_temperature))
test_probs_calibrated = 1 / (1 + np.exp(-test_logits / optimal_temperature))

topk_compare_fracs = (0.01, 0.02, 0.05, 0.10)
topk_before = compute_topk_lift_table(all_labels, all_probs_test, topk_compare_fracs)
topk_after = compute_topk_lift_table(all_labels, test_probs_calibrated, topk_compare_fracs)
prec_before, rec_before, _ = precision_recall_curve(all_labels, all_probs_test)
prec_after, rec_after, _ = precision_recall_curve(all_labels, test_probs_calibrated)
pr_auc_calibrated = auc(rec_after, prec_after)

print('\n' + '=' * 80)
print('PREDICTION DISTRIBUTION: Before vs After Calibration')
print('=' * 80)
print('\nVALIDATION SET:')
print(f"  Before: {((val_probs_original < 0.01) | (val_probs_original > 0.99)).sum()} / {len(val_probs_original)} extreme predictions ({((val_probs_original < 0.01) | (val_probs_original > 0.99)).mean():.1%})")
print(f"  After:  {((val_probs_calibrated < 0.01) | (val_probs_calibrated > 0.99)).sum()} / {len(val_probs_calibrated)} extreme predictions ({((val_probs_calibrated < 0.01) | (val_probs_calibrated > 0.99)).mean():.1%})")
print('\nTEST SET:')
print(f"  Before: {((all_probs_test < 0.01) | (all_probs_test > 0.99)).sum()} / {len(all_probs_test)} extreme predictions ({((all_probs_test < 0.01) | (all_probs_test > 0.99)).mean():.1%})")
print(f"  After:  {((test_probs_calibrated < 0.01) | (test_probs_calibrated > 0.99)).sum()} / {len(test_probs_calibrated)} extreme predictions ({((test_probs_calibrated < 0.01) | (test_probs_calibrated > 0.99)).mean():.1%})")

print('\n' + '=' * 80)
print('RANKING COMPARISON')
print('=' * 80)
print(f'Original Test PR-AUC:    {pr_auc:.4f}')
print(f'Calibrated Test PR-AUC:  {pr_auc_calibrated:.4f}')
print(f'Top-5% lift before:      {topk_before.loc[np.isclose(topk_before["top_pct"], 5.0), "lift"].iloc[0]:.4f}')
print(f'Top-5% lift after:       {topk_after.loc[np.isclose(topk_after["top_pct"], 5.0), "lift"].iloc[0]:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(all_probs_test, bins=50, alpha=0.5, label='Before', edgecolor='black')
axes[0].hist(test_probs_calibrated, bins=50, alpha=0.5, label='After', edgecolor='black')
axes[0].set(xlabel='Predicted Probability', ylabel='Count', title='Prediction Distribution')
axes[0].legend()
axes[0].grid(alpha=0.3)

fraction_of_positives_before, mean_predicted_value_before = calibration_curve(
    all_labels, all_probs_test, n_bins=10, strategy='uniform'
)
fraction_of_positives_after, mean_predicted_value_after = calibration_curve(
    all_labels, test_probs_calibrated, n_bins=10, strategy='uniform'
)
axes[1].plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
axes[1].plot(mean_predicted_value_before, fraction_of_positives_before, marker='o', label='Before (T=1.0)', linewidth=2)
axes[1].plot(mean_predicted_value_after, fraction_of_positives_after, marker='s', label=f'After (T={optimal_temperature:.2f})', linewidth=2)
axes[1].set(xlabel='Mean Predicted Probability', ylabel='Fraction of Positives', title='Calibration Curve')
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(topk_before['top_pct'], topk_before['lift'], marker='o', lw=2, label='Before')
axes[2].plot(topk_after['top_pct'], topk_after['lift'], marker='s', lw=2, linestyle='--', label=f'After (T={optimal_temperature:.2f})')
axes[2].axhline(1.0, color='grey', linestyle='--', alpha=0.6)
axes[2].set(title='Top-K Lift Comparison', xlabel='Top scored customers (%)', ylabel='Lift vs. baseline')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('temperature_scaling_calibration.png', dpi=150, bbox_inches='tight')
print('\n✓ Calibration plots saved to temperature_scaling_calibration.png')
plt.show()

print('\n' + '=' * 80)
print('✓ Temperature scaling complete!')
print(f'  Use temperature={optimal_temperature:.4f} in inference to get calibrated probabilities')
print('=' * 80)


In [ ]:
print("\n" + "=" * 80)
print("TEMPERATURE OPTIONS FOR DIFFERENT USE CASES")
print("=" * 80)

print(f"""
The optimal temperature T={optimal_temperature:.4f} minimizes calibration error (NLL),
but makes predictions MORE extreme (higher confidence).

For the DEMO application, you may want a higher temperature to prevent
rapid score jumps when adding transactions:
""")

# Test different temperatures
demo_temperatures = [1.0, 2.0, 3.0, 5.0]
print(f"\n{'Temperature':<15} {'Extreme %':<15} {'Mean Prob':<15} {'Std Prob':<15} {'Use Case'}")
print("-" * 85)

# Original optimized temperature
test_probs_opt = 1 / (1 + np.exp(-test_logits / optimal_temperature))
extreme_pct_opt = ((test_probs_opt < 0.01) | (test_probs_opt > 0.99)).mean()
print(f"{optimal_temperature:<15.4f} {extreme_pct_opt:<15.1%} {test_probs_opt.mean():<15.4f} {test_probs_opt.std():<15.4f} Best Calibration")

for T in demo_temperatures:
    test_probs_T = 1 / (1 + np.exp(-test_logits / T))
    extreme_pct = ((test_probs_T < 0.01) | (test_probs_T > 0.99)).mean()
    use_case = ""
    if T == 1.0:
        use_case = "Original Model"
    elif T == 2.0:
        use_case = "Smoother (Recommended for Demo)"
    elif T == 3.0:
        use_case = "Very Smooth"
    elif T == 5.0:
        use_case = "Extremely Smooth"
    print(f"{T:<15.2f} {extreme_pct:<15.1%} {test_probs_T.mean():<15.4f} {test_probs_T.std():<15.4f} {use_case}")

print(f"""
\n**Recommendation:**
- Use T={optimal_temperature:.4f} for production risk scoring (best calibration)
- Use T=2.0 or T=3.0 for the demo UI (smoother, less jumpy predictions)

Higher temperature = smoother predictions, less sensitive to small changes.
""")

# Visualize the effect of different temperatures
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sample a subset for visualization (fraud and non-fraud)
fraud_idx = np.where(all_labels == 1)[0][:100]
legit_idx = np.where(all_labels == 0)[0][:100]
sample_idx = np.concatenate([fraud_idx, legit_idx])
sample_logits = test_logits[sample_idx]
sample_labels = all_labels[sample_idx]

# Show predictions for different temperatures
temperatures_to_plot = [optimal_temperature, 1.0, 2.0, 3.0]
colors = ['red', 'blue', 'green', 'orange']

for T, color in zip(temperatures_to_plot, colors):
    probs_T = 1 / (1 + np.exp(-sample_logits / T))
    axes[0].scatter(range(len(probs_T)), probs_T, alpha=0.3, s=20, 
                   label=f'T={T:.2f}', color=color)

axes[0].scatter(range(len(sample_labels)), sample_labels, 
               marker='x', s=100, color='black', label='True Label', alpha=0.5)
axes[0].set(xlabel='Sample Index', ylabel='Predicted Probability',
           title='Effect of Temperature on Predictions\n(First 100 fraud + 100 legit)',
           ylim=[-0.1, 1.1])
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.3)

# Distribution comparison
for T, color in zip([optimal_temperature, 2.0], ['red', 'green']):
    probs_T = 1 / (1 + np.exp(-test_logits / T))
    axes[1].hist(probs_T, bins=50, alpha=0.4, label=f'T={T:.2f}', 
                color=color, edgecolor='black')

axes[1].set(xlabel='Predicted Probability', ylabel='Count',
           title=f'Distribution: Optimal (T={optimal_temperature:.2f}) vs Demo (T=2.0)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('temperature_comparison.png', dpi=150, bbox_inches='tight')
print("✓ Temperature comparison saved to temperature_comparison.png")
plt.show()

# Save both temperatures for different use cases
TEMPERATURE_CALIBRATED = optimal_temperature  # For production
TEMPERATURE_DEMO = 2.0  # For smoother UI experience

print(f"\n✓ Saved two temperature settings:")
print(f"  TEMPERATURE_CALIBRATED = {TEMPERATURE_CALIBRATED:.4f} (for production)")
print(f"  TEMPERATURE_DEMO = {TEMPERATURE_DEMO:.1f} (for demo/UI)")


## Evaluation Summary

The notebook now emphasizes **ranking quality** and **pseudo-label confidence dynamics**:

- **PR-AUC** remains available as a secondary ranking metric.
- **Top-K lift** is the primary operational metric for how concentrated fraud is in the highest-risk customers.
- **Confidence stability** tracks whether pseudo-labeling pushes scores away from 0.5 and toward 0/1 across rounds.
- **Leave-one-out feature ablation** measures how much each final customer-node feature contributes when removed.

Temperature scaling remains optional post-processing for score calibration, but the main model evaluation is now centered on lift, stability, and feature sensitivity.


# Save Model Artifacts

We save everything needed for **incremental inference**:
- `fraud_sage_model.pth` — model weights + training metadata
- `sage_artifacts.pkl` — customer feature scaler, node-ID mappings, feature column list, graph topology (edge tensors)
- `model_output.csv` — risk scores for all KYC customers

At demo time, adding a new customer only requires:
1. Compute their 13-feature vector → scale with `cust_scaler`
2. Append a new row to the graph's `x_customer` tensor
3. Add edges to their category/city nodes
4. Run one `NeighborLoader` batch → instant risk score


In [ ]:
import pickle
import torch

# ── 1. Model weights + training metadata ─────────────────────────────────────
sage_model_path = 'fraud_sage_model.pth'
topk_lift_summary = {
    f"top_{int(row['top_pct'])}pct_lift": float(row['lift'])
    for _, row in topk_lift_df.iterrows()
}
confidence_stability_summary = {
    'initial_mean_confidence_margin': float(confidence_round_df['mean_confidence_margin'].iloc[0]),
    'final_mean_confidence_margin': float(confidence_round_df['mean_confidence_margin'].iloc[-1]),
    'initial_uncertain_share_40_60': float(confidence_round_df['uncertain_share_40_60'].iloc[0]),
    'final_uncertain_share_40_60': float(confidence_round_df['uncertain_share_40_60'].iloc[-1]),
    'initial_confident_share_10_90': float(confidence_round_df['confident_share_10_90'].iloc[0]),
    'final_confident_share_10_90': float(confidence_round_df['confident_share_10_90'].iloc[-1]),
}

torch.save({
    'model_state_dict': model_sage.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': NUM_EPOCHS,
    'pr_auc': pr_auc,
    'fraud_recall': fr,
    'fraud_precision': fp2,
    'fraud_f1': ff1,
    'topk_lift': topk_lift_summary,
    'confidence_stability': confidence_stability_summary,
    'pseudo_label_history': pseudo_label_history,
    'confidence_round_history': confidence_round_history,
    'leave_one_out_results': leave_one_out_results.to_dict(orient='records'),
    'total_pseudo_labeled': total_pseudo_labeled,
    'temperature_calibrated': TEMPERATURE_CALIBRATED,
    'temperature_demo': TEMPERATURE_DEMO,
    'model_config': {
        'hidden_channels': HIDDEN_CHANNELS,
        'customer_feature_cols': CUSTOMER_FEATURE_COLS,
        'transaction_feature_cols': TRANSACTION_FEATURE_COLS,
        'num_customer_features': len(CUSTOMER_FEATURE_COLS),
    },
    'training_config': {
        'lr': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
        'num_epochs': NUM_EPOCHS, 'focal_alpha': FOCAL_ALPHA,
        'focal_gamma': FOCAL_GAMMA, 'pseudo_labeling': ENABLE_PL,
        'fraud_threshold': FRAUD_THRESHOLD, 'legit_threshold': LEGIT_THRESHOLD,
        'warmup_epochs': WARMUP_EPOCHS,
        'num_neighbors': NUM_NEIGHBORS, 'batch_size': BATCH_SIZE,
        'split_node_type': 'customer',
    },
    'label_config': {
        'node_type': 'customer',
        'label_column': 'fraud_label',
        'fraud': 1,
        'legit': 0,
        'unlabeled': -1,
    },
}, sage_model_path)
print(f"✓ Model saved → {sage_model_path}")
print(f"  - PR-AUC: {pr_auc:.4f}")
print(f"  - Top-5% lift: {primary_topk_lift:.4f}")
print(f"  - temperature_calibrated: {TEMPERATURE_CALIBRATED:.4f} (for production)")
print(f"  - temperature_demo: {TEMPERATURE_DEMO:.1f} (for demo UI)")

# ── 2. Inference artifacts (scaler + mappings + graph topology) ───────────────
# We save the edge tensors so we can add new nodes at inference time
sage_artifacts = {
    # Feature engineering
    'cust_scaler': cust_scaler,
    'customer_feature_cols': CUSTOMER_FEATURE_COLS,
    'transaction_feature_cols': TRANSACTION_FEATURE_COLS,
    'num_customer_features': len(CUSTOMER_FEATURE_COLS),
    # Node / label semantics
    'node_type': 'customer',
    'label_column': 'fraud_label',
    'label_values': {'fraud': 1, 'legit': 0, 'unlabeled': -1},
    # Node ID mappings
    'cust_map': cust_map,
    'cat_map': cat_map,
    'city_map': city_map,
    'num_cust': num_cust,
    'num_cat': num_cat,
    'num_city': num_city,
    # Graph topology (static hub nodes never change between deployments)
    'x_cat': x_cat,
    'x_city': x_city,
    # Full edge tensors (for NeighborLoader context around new nodes)
    'edge_cust_cat': data_sage['customer', 'purchases_at', 'category'].edge_index,
    'edge_cust_city': data_sage['customer', 'transacts_in', 'city'].edge_index,
    # Customer-node features / labels
    'x_cust': x_cust,
    'customer_y': data_sage['customer'].y,
    # Hyperparameters needed for NeighborLoader at inference
    'num_neighbors': NUM_NEIGHBORS,
    'hidden_channels': HIDDEN_CHANNELS,
    # Evaluation metadata
    'topk_lift': topk_lift_summary,
    'confidence_stability': confidence_stability_summary,
    'leave_one_out_results': leave_one_out_results.to_dict(orient='records'),
    # Temperature scaling for calibration
    'temperature_calibrated': TEMPERATURE_CALIBRATED,
    'temperature_demo': TEMPERATURE_DEMO,
}

with open('sage_artifacts.pkl', 'wb') as f:
    pickle.dump(sage_artifacts, f, protocol=4)
print('✓ Artifacts saved → sage_artifacts.pkl')

# ── 3. Risk scores for all KYC customers ─────────────────────────────────────
model_sage.eval()
all_customer_node_probs = collect_all_customer_probs(model_sage, data_graph)

kyc_ind = pd.read_csv(os.path.join(data_path, 'kyc_individual.csv.gz'), usecols=['customer_id'])
kyc_biz = pd.read_csv(os.path.join(data_path, 'kyc_smallbusiness.csv.gz'), usecols=['customer_id'])
kyc_ids = pd.concat([kyc_ind['customer_id'], kyc_biz['customer_id']]).astype(str).unique()

probs_np = all_customer_node_probs.cpu().numpy()
output_df_sage = pd.DataFrame({'customer_id': kyc_ids})
output_df_sage['cust_idx'] = output_df_sage['customer_id'].map(cust_map)
output_df_sage['risk_score'] = output_df_sage['cust_idx'].apply(
    lambda i: float(probs_np[int(i)]) if pd.notna(i) else 0.0
)
output_df_sage['predicted_label'] = (output_df_sage['risk_score'] > 0.5).astype(int)
output_df_sage[['customer_id', 'predicted_label', 'risk_score']].to_csv('model_output.csv', index=False)
print('✓ Customer-node predictions saved → model_output.csv')

hr = (output_df_sage['risk_score'] > 0.8).sum()
mr = ((output_df_sage['risk_score'] > 0.5) & (output_df_sage['risk_score'] <= 0.8)).sum()
lr2 = (output_df_sage['risk_score'] <= 0.2).sum()
print(f"\nRisk distribution ({len(output_df_sage):,} KYC customers):")
print(f"  HIGH  (>80%): {hr:>6,} ({hr / len(output_df_sage) * 100:.2f}%)")
print(f"  MED (50-80%): {mr:>6,} ({mr / len(output_df_sage) * 100:.2f}%)")
print(f"  LOW  (<20%):  {lr2:>6,} ({lr2 / len(output_df_sage) * 100:.2f}%)")


## Temperature Scaling Applied

The saved artifacts still include optional temperature parameters for downstream scoring:

1. **`temperature_calibrated`** — learned on validation NLL for calibrated probabilities.
2. **`temperature_demo`** — a smoother fallback for UI-style scoring when less abrupt probability changes are preferred.

These values only affect post-hoc probability scaling. The core notebook evaluation now comes from PR-AUC, Top-K lift, confidence stability, and leave-one-out ablation.


# Inductive Inference — Scoring New Nodes at Demo Time

This is the payoff of GraphSAGE: we can add a brand-new customer (one the model has **never seen**) to the graph, and score them instantly using the learned neighborhood-aggregation function.

**No retraining. No graph rebuild. Just feature computation + one mini-batch forward pass.**

```
  new_customer_features  →  scale  →  append to x_cust
  new transactions       →  add edges to category/city nodes
  NeighborLoader(new_node_id)  →  model forward  →  risk score
```


In [ ]:
def score_new_customers(
    new_customer_records: list,
    model,
    artifacts: dict,
    device=None,
) -> pd.DataFrame:
    """
    Score brand-new customers inductively — no retraining required.

    Parameters
    ----------
    new_customer_records : list of dict
        Each dict must have keys matching CUSTOMER_FEATURE_COLS:
        ['age', 'income', 'tenure', 'sales', 'emp_count', 'is_biz',
         'avg_txn_amount', 'max_txn_amount', 'std_txn_amount', 'txn_count',
         'cash_rate', 'ecom_rate', 'avg_24h_velocity', 'unique_cities',
         'unique_categories', 'min_time_delta', 'time_span_hours', 'geo_velocity']
        Plus optionally:
        'customer_id'  — identifier (default: "NEW_{i}")
        'merchant_categories' — list of category names the customer transacts at
        'cities'              — list of city names the customer transacts in

    Returns
    -------
    pd.DataFrame with columns: customer_id, risk_score, risk_tier, predicted_label
    """
    if device is None:
        device = next(model.parameters()).device

    scaler   = artifacts['cust_scaler']
    feat_cols = artifacts['customer_feature_cols']
    cat_map_a = artifacts['cat_map']
    city_map_a = artifacts['city_map']
    x_cust_base = artifacts['x_cust']
    x_cat_base  = artifacts['x_cat']
    x_city_base = artifacts['x_city']
    ei_cc  = artifacts['edge_cust_cat']
    ei_cct = artifacts['edge_cust_city']
    num_neighbors = artifacts['num_neighbors']
    temperature = artifacts.get('temperature_demo', 1.0)
    N_existing = x_cust_base.shape[0]

    # ── Step 1: Compute feature vectors for new customers ────────────────────
    new_feats = []
    for rec in new_customer_records:
        row = [float(rec.get(c, 0.0)) for c in feat_cols]
        new_feats.append(row)

    new_feats_raw   = np.array(new_feats, dtype=np.float32)
    new_feats_scaled = scaler.transform(new_feats_raw)
    new_feats_scaled = np.nan_to_num(new_feats_scaled, nan=0.0, posinf=3.0, neginf=-3.0)
    x_new = torch.tensor(new_feats_scaled, dtype=torch.float)

    # ── Step 2: Build augmented node feature tensors ─────────────────────────
    x_cust_aug = torch.cat([x_cust_base, x_new], dim=0)
    new_node_ids = list(range(N_existing, N_existing + len(new_customer_records)))

    # ── Step 3: Build edges from new customers to their categories/cities ────
    new_cat_edges_src, new_cat_edges_dst   = [], []
    new_city_edges_src, new_city_edges_dst = [], []

    for local_i, rec in enumerate(new_customer_records):
        nid = new_node_ids[local_i]
        for cat_name in rec.get('merchant_categories', []):
            cat_idx_val = cat_map_a.get(cat_name)
            if cat_idx_val is not None:
                new_cat_edges_src.append(nid)
                new_cat_edges_dst.append(cat_idx_val)
        for city_name in rec.get('cities', []):
            city_idx_val = city_map_a.get(city_name.upper() if city_name else 'UNKNOWN')
            if city_idx_val is not None:
                new_city_edges_src.append(nid)
                new_city_edges_dst.append(city_idx_val)

    # Merge new edges with existing
    if new_cat_edges_src:
        extra_cc = torch.tensor([new_cat_edges_src, new_cat_edges_dst], dtype=torch.long)
        ei_cc_aug = torch.cat([ei_cc, extra_cc], dim=1)
    else:
        ei_cc_aug = ei_cc

    if new_city_edges_src:
        extra_city = torch.tensor([new_city_edges_src, new_city_edges_dst], dtype=torch.long)
        ei_cct_aug = torch.cat([ei_cct, extra_city], dim=1)
    else:
        ei_cct_aug = ei_cct

    # ── Step 4: Construct a temporary HeteroData graph ───────────────────────
    from torch_geometric.data import HeteroData
    import torch_geometric.transforms as T

    tmp = HeteroData()
    tmp['customer'].x = x_cust_aug
    tmp['category'].x = x_cat_base
    tmp['city'].x     = x_city_base

    tmp['customer', 'purchases_at', 'category'].edge_index = ei_cc_aug
    tmp['customer', 'transacts_in', 'city'].edge_index     = ei_cct_aug
    tmp = T.ToUndirected()(tmp)

    # ── Step 5: NeighborLoader focused on the new nodes ──────────────────────
    seed_mask = torch.zeros(x_cust_aug.shape[0], dtype=torch.bool)
    for nid in new_node_ids:
        seed_mask[nid] = True

    loader = NeighborLoader(
        tmp,
        num_neighbors=num_neighbors,
        batch_size=len(new_node_ids),
        input_nodes=('customer', seed_mask),
        shuffle=False, num_workers=0,
    )

    # ── Step 6: Forward pass ──────────────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        batch = next(iter(loader)).to(device)
        logits = model(batch.x_dict, batch.edge_index_dict)
        probs = torch.sigmoid(logits.squeeze()[:len(new_node_ids)] / temperature).cpu().numpy()

    probs = np.atleast_1d(probs)

    # ── Step 7: Build results DataFrame ──────────────────────────────────────
    results = []
    for i, rec in enumerate(new_customer_records):
        score = float(probs[i])
        tier  = 'HIGH' if score >= 0.70 else ('MEDIUM' if score >= 0.40 else 'LOW')
        results.append({
            'customer_id':     rec.get('customer_id', f'NEW_{i}'),
            'risk_score':      round(score, 6),
            'risk_tier':       tier,
            'predicted_label': int(score >= 0.5),
        })
    return pd.DataFrame(results)


# ── Demo: score 3 synthetic new customers ────────────────────────────────────
demo_new_customers = [
    {
        'customer_id': 'DEMO_high_risk',
        # High-risk profile: high velocity, tight timing, multi-city cash activity
        'age': 28, 'income': 35000, 'tenure': 90, 'sales': 0, 'emp_count': 0, 'is_biz': 0,
        'avg_txn_amount': 4200, 'max_txn_amount': 18000, 'std_txn_amount': 3800, 'txn_count': 120,
        'cash_rate': 0.45, 'ecom_rate': 0.05, 'avg_24h_velocity': 8.5,
        'unique_cities': 3, 'unique_categories': 6, 'min_time_delta': 2, 'time_span_hours': 5, 'geo_velocity': 0.6,
        'merchant_categories': ['es_tech', 'es_contents'],
        'cities': ['TORONTO', 'MONTREAL', 'CALGARY'],
    },
    {
        'customer_id': 'DEMO_normal',
        # Typical low-risk retail customer
        'age': 45, 'income': 85000, 'tenure': 2920, 'sales': 0, 'emp_count': 0, 'is_biz': 0,
        'avg_txn_amount': 120, 'max_txn_amount': 600, 'std_txn_amount': 80, 'txn_count': 42,
        'cash_rate': 0.02, 'ecom_rate': 0.20, 'avg_24h_velocity': 1.1,
        'unique_cities': 1, 'unique_categories': 4, 'min_time_delta': 180, 'time_span_hours': 720, 'geo_velocity': 0,
        'merchant_categories': ['es_food', 'es_transportation'],
        'cities': ['TORONTO'],
    },
    {
        'customer_id': 'DEMO_new_biz',
        # New small business with no transaction history yet
        'age': 38, 'income': 150000, 'tenure': 30, 'sales': 500000, 'emp_count': 12, 'is_biz': 1,
        'avg_txn_amount': 0, 'max_txn_amount': 0, 'std_txn_amount': 0, 'txn_count': 0,
        'cash_rate': 0, 'ecom_rate': 0, 'avg_24h_velocity': 0,
        'unique_cities': 0, 'unique_categories': 0, 'min_time_delta': 0, 'time_span_hours': 0, 'geo_velocity': 0,
        'merchant_categories': [],
        'cities': [],
    },
]

print("Scoring 3 new customers inductively (no retraining)...\n")
results_df = score_new_customers(
    demo_new_customers,
    model=model_sage,
    artifacts={
        'cust_scaler': cust_scaler,
        'customer_feature_cols': CUSTOMER_FEATURE_COLS,
        'cat_map': cat_map, 'city_map': city_map,
        'x_cust': x_cust, 'x_cat': x_cat, 'x_city': x_city,
        'edge_cust_cat':  data_sage['customer', 'purchases_at', 'category'].edge_index,
        'edge_cust_city': data_sage['customer', 'transacts_in', 'city'].edge_index,
        'num_neighbors': NUM_NEIGHBORS,
        'hidden_channels': HIDDEN_CHANNELS,
        'temperature_demo': TEMPERATURE_DEMO,
    },
    device=device,
)


# Interpretability — RF Proxy + SHAP

GraphSAGE is a black box. To produce **human-readable explanations** we train a Random Forest to *mimic* the GraphSAGE risk scores on the 13 semantic features, then use SHAP to explain *why* each customer got their score.

Because our customer node features are already semantically meaningful (age, income, velocity, etc.), the RF explanation maps cleanly to analyst language — no opaque embedding dimensions.


In [ ]:
import shap
import warnings
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings('ignore', category=UserWarning)

FEATURE_NAMES_SAGE = CUSTOMER_FEATURE_COLS   # customer-node feature schema from training
FEATURE_DESCRIPTIONS_SAGE = {
    'age':              'customer age',
    'income':           'annual income',
    'tenure':           'account tenure (days)',
    'sales':            'annual sales (businesses)',
    'emp_count':        'employee headcount',
    'is_biz':           'business account type',
    'avg_txn_amount':   'average transaction amount',
    'max_txn_amount':   'maximum single transaction',
    'std_txn_amount':   'transaction amount variability',
    'txn_count':        'total transaction volume',
    'cash_rate':        'cash withdrawal frequency',
    'ecom_rate':        'e-commerce transaction rate',
    'avg_24h_velocity': '24-hour transaction velocity',
    'unique_cities':    'unique transaction cities',
    'unique_categories':'unique merchant categories',
    'min_time_delta':   'min minutes between transactions',
    'time_span_hours':  'transaction window (hours)',
    'geo_velocity':     'geographic velocity (cities/hr)',
}

# ── Align GraphSAGE risk scores to the customer feature matrix (by cust_idx) ──
probs_np_sage = all_customer_node_probs.cpu().numpy()

# cp already has cust_idx sorted 0..N; use raw (unscaled) features for RF
cp_sorted = cp.sort_values('cust_idx').reset_index(drop=True)
X_rf = cp_sorted[FEATURE_NAMES_SAGE].values.astype(np.float32)
cust_idxs_rf = cp_sorted['cust_idx'].values.astype(int)
valid_rf = cust_idxs_rf < len(probs_np_sage)

y_risk_rf = np.zeros(len(cp_sorted), dtype=np.float32)
y_risk_rf[valid_rf] = probs_np_sage[cust_idxs_rf[valid_rf]]

# ── Train RF proxy ────────────────────────────────────────────────────────────
print("Training RF proxy on customer features → GraphSAGE risk scores...")
rf_sage = RandomForestRegressor(
    n_estimators=300, max_depth=8, min_samples_leaf=5,
    n_jobs=-1, random_state=42,
)
rf_sage.fit(X_rf, y_risk_rf)

# R² vs hard-labeled customers
y_true_sage = data_sage['customer'].y.cpu().numpy()
hard_lm = np.array([
    valid_rf[i] and int(cust_idxs_rf[i]) < len(y_true_sage) and y_true_sage[int(cust_idxs_rf[i])] != -1
    for i in range(len(cp_sorted))
])
rf_r2 = rf_sage.score(X_rf[hard_lm], y_risk_rf[hard_lm])
print(f"RF R² on hard-labeled subset: {rf_r2:.4f}")

fi = pd.Series(rf_sage.feature_importances_, index=FEATURE_NAMES_SAGE).sort_values(ascending=False)
print("\nTop feature importances (RF proxy):")
print(fi.head(10).to_string())

# ── SHAP — pre-compute for all customers (single batch call) ─────────────────
print("\nComputing SHAP values for all customers...")
explainer_sage   = shap.TreeExplainer(rf_sage)
shap_values_sage = explainer_sage.shap_values(X_rf)
shap_base_sage   = float(np.atleast_1d(explainer_sage.expected_value)[0])

print(f"SHAP matrix: {shap_values_sage.shape}  |  base value: {shap_base_sage:.4f}")
print("✓ SHAP pre-computation complete")


In [ ]:
import joblib
import json

# ── Save RF artifacts ─────────────────────────────────────────────────────────
rf_dir = "rf_model_sage"
os.makedirs(rf_dir, exist_ok=True)

joblib.dump(rf_sage,          os.path.join(rf_dir, "rf_proxy.joblib"))
joblib.dump(explainer_sage,   os.path.join(rf_dir, "shap_explainer.joblib"))

meta_sage = {
    "feature_names":    FEATURE_NAMES_SAGE,
    "shap_base_value":  shap_base_sage,
    "rf_r2":            rf_r2,
    "model_type":       "GraphSAGE",
}
with open(os.path.join(rf_dir, "meta.json"), "w") as f:
    json.dump(meta_sage, f, indent=2)

print(f"RF artifacts saved to '{rf_dir}/'")

# ── Explanation builder ───────────────────────────────────────────────────────
THRESH_HIGH   = 0.70
THRESH_MEDIUM = 0.40

def _risk_tier(score):
    return 'HIGH' if score >= THRESH_HIGH else ('MEDIUM' if score >= THRESH_MEDIUM else 'LOW')

def _top_shap_drivers(shap_row, n=3):
    pos_idx = np.where(shap_row > 0)[0]
    if len(pos_idx) == 0:
        pos_idx = np.argsort(shap_row)[-n:][::-1]
    else:
        pos_idx = pos_idx[np.argsort(shap_row[pos_idx])[::-1]][:n]
    return [{'feature': FEATURE_NAMES_SAGE[i],
             'description': FEATURE_DESCRIPTIONS_SAGE.get(FEATURE_NAMES_SAGE[i], FEATURE_NAMES_SAGE[i]),
             'shap_value': float(shap_row[i])} for i in pos_idx]

def _build_narrative(risk_tier, risk_score, shap_drivers):
    if risk_tier == 'LOW':
        return (f"Customer presents a LOW risk profile (score {risk_score:.1%}). "
                "No significant behavioural anomalies detected.")
    drivers_text = ', '.join(d['description'] for d in shap_drivers)
    if risk_tier == 'HIGH':
        heading = f"⚠ HIGH RISK — score {risk_score:.1%}"
        action  = "Recommend immediate case review and enhanced due-diligence."
    else:
        heading = f"⚡ MEDIUM RISK — score {risk_score:.1%}"
        action  = "Recommend monitoring and secondary review."
    return f"{heading}. Primary behavioural drivers: {drivers_text}. {action}"


# ── Build explanation export ──────────────────────────────────────────────────
# Map output_df_sage customer_id → row in cp_sorted for fast lookup
id_to_row_sage = {str(cid): i for i, cid in enumerate(cp_sorted['customer_id'])}

records_sage = []
for _, row in output_df_sage.iterrows():
    cid   = str(row['customer_id'])
    score = float(row['risk_score'])
    tier  = _risk_tier(score)
    ridx  = id_to_row_sage.get(cid)

    if ridx is None:
        records_sage.append({'customer_id': cid, 'risk_score': score, 'risk_tier': tier,
                              'predicted_label': int(row['predicted_label']),
                              'narrative': 'not in feature matrix',
                              'driver_1': '', 'driver_2': '', 'driver_3': ''})
        continue

    drivers  = _top_shap_drivers(shap_values_sage[ridx], n=3)
    narrative = _build_narrative(tier, score, drivers)
    records_sage.append({
        'customer_id':     cid,
        'risk_score':      score,
        'risk_tier':       tier,
        'predicted_label': int(row['predicted_label']),
        'narrative':       narrative,
        'driver_1':        drivers[0]['description'] if len(drivers) > 0 else '',
        'driver_1_shap':   round(drivers[0]['shap_value'], 5) if len(drivers) > 0 else '',
        'driver_2':        drivers[1]['description'] if len(drivers) > 1 else '',
        'driver_2_shap':   round(drivers[1]['shap_value'], 5) if len(drivers) > 1 else '',
        'driver_3':        drivers[2]['description'] if len(drivers) > 2 else '',
        'driver_3_shap':   round(drivers[2]['shap_value'], 5) if len(drivers) > 2 else '',
    })

explanations_sage = pd.DataFrame(records_sage)
explanations_sage.to_csv('model_output_explanations_sage.csv', index=False)
print(f"✓ Saved explanations → model_output_explanations_sage.csv ({len(explanations_sage):,} rows)")

for t in ['HIGH', 'MEDIUM', 'LOW']:
    n = (explanations_sage['risk_tier'] == t).sum()
    print(f"  {t:<8}: {n:>6,}  ({n/len(explanations_sage)*100:.2f}%)")

import gdown
import os

explanation_file = "model_output_explanations.csv"

if os.path.exists(explanation_file):
    print(f"{explanation_file} found. Skipping download.")
else:
    print(f"{explanation_file} missing. Downloading from Google Drive...")
    url = "https://drive.google.com/file/d/1_5RZnzKwspT0-WQTTOxwWT5SXPf0CRYr/view?usp=sharing"
    gdown.download(url, explanation_file, fuzzy=True)
    print(f"Downloaded {explanation_file}.")

print("LLM Generated Explanation file ready.")


In [ ]:
!streamlit run ./Home.py
# Or run in terminal: streamlit run Home.py